In [1]:
from deck import full_euchre_deck
from dealer import Dealer
# from numba import njit
from n_branches import array_set_difference
from n_play_round import round1, next_round
from tree_search import definitive_winner, find_best_opener, n_trick_sim
import numpy as np


In [2]:
# game = Dealer(deck=full_euchre_deck ,players=4)
# # game.stack_deck(stack_cards=dealer_pick, player=4)
# game.deal_cards()
# hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
# hands5

In [3]:
hands_test = np.array([[[-10,   0],
        [-11,   0],
        [-14,   0],
        [-12,   0],
        [  0, -10]],

       [[  0, -12],
        [  0, -13],
        [  0, -14],
        [-13,   0],
        [  0, 100]],

       [[ 14,   0],
        [  0, 135],
        [ 10,   0],
        [  0,  90],
        [  0,  -9]],

       [[ 12,   0],
        [  0, 140],
        [ 11,   0],
        [  9,   0],
        [  0, 110]]])

In [4]:
starting_lead = 0
previous_winners = np.array([])

In [5]:
best_opener = find_best_opener(lead=starting_lead, hands=hands_test, tricks=5, previous_winners=np.array([]), sim_func=n_trick_sim, verbose=True)

[0.08461393 0.09096482 0.23288284 0.09096482 0.07961334]


In [6]:
r2_leads, r2_hands = round1(
    hands_dealt=hands_test, chosen_card=best_opener, leader=starting_lead
)

In [10]:
for i in range(1,4):
    print(i)

1
2
3


In [11]:
1 % 4

1

In [ ]:
# i think something like this could work to recursively search for the optimal responses.

for player in range(1,4):
    responder = (starting_lead + player) % 4
    responder_team = responder % 2
    possible_plays_responder = np.unique(r2_hands[:, responder], axis=0)
    for target in possible_plays_responder:
        mask = np.all(r2_hands[:, responder] == target, axis=(1, 2))
        filtered_hands = r2_hands[mask]
        filtered_leads = r2_leads[mask] 

        r3_leads, r3_hands, r3_score = next_round(
            current_hands=filtered_hands,
            leads=filtered_leads,
            game_round=2,
            game_score=filtered_leads.reshape(-1, 1)
        )
        r4_leads, r4_hands, r4_score = next_round(
            current_hands=r3_hands,
            leads=r3_leads,
            game_round=3,
            game_score=r3_score
        )
        r5_leads, r5_hands, r5_score = next_round(
            current_hands=r4_hands,
            leads=r4_leads,
            game_round=4,
            game_score=r4_score
        )
        r6_leads, r6_hands, r6_score = next_round(
            current_hands=r5_hands,
            leads=r5_leads,
            game_round=5,
            game_score=r5_score
        )
        results = r6_score.reshape(r6_score.shape[0], 5)
    
    # Calculate meta results
        meta_results = np.zeros(results.shape[0], dtype=np.int64)
    
        for i in range(len(results)):
            # Count wins for odd-numbered players (team 1)
            total_odd_wins = np.sum(results[i] % 2)
            
            # Add previous winners to the count
            if len(previous_winners) > 0:
                total_odd_wins += np.sum(previous_winners % 2)
            
            # Team wins if they get 3+ tricks out of 5 total
            if total_odd_wins >= 3:
                score = 0
            else: 
                score = 1

            meta_results[i] = score

        print(np.mean(meta_results), responder, target)



0.23288284256690236 1 [[  0 -12]
 [  0 -13]
 [  0 -14]
 [  0 100]]
0.11525704809286899 2 [[ 14   0]
 [  0 135]
 [ 10   0]
 [  0  -9]]
0.33976470588235297 2 [[ 14   0]
 [  0 135]
 [ 10   0]
 [  0  90]]
0.06621961441743504 2 [[14  0]
 [10  0]
 [ 0 90]
 [ 0 -9]]
0.34560158441987127 3 [[ 12   0]
 [  0 140]
 [ 11   0]
 [  0 110]]
0.08537549407114625 3 [[ 12   0]
 [  0 140]
 [ 11   0]
 [  9   0]]
0.1130030959752322 3 [[ 12   0]
 [ 11   0]
 [  9   0]
 [  0 110]]


In [8]:
# it appears that the function for best response relies on the opposing team making mistakes. Need to find a fix for this